In [23]:
from pymilvus import MilvusClient
from embedding import query_vectorisation
import json
import numpy as np
from tqdm import tqdm
from  time import time
import pandas as pd
from langdetect import detect

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cosine
from sklearn.metrics import ndcg_score
from sentence_transformers import SentenceTransformer

In [34]:
client = MilvusClient("../../rag_v1_milvus.db")
collection = "rag_v1"

## Exact Match (TF-IDF / BM25)

In [22]:
model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")

In [24]:
def exact_match_retrieval_tfidf(query: str, documents: list, model) -> dict:
    start_time = time.time()
    corpus = [doc["text"] for doc in documents]
    langs = [doc["lang"] for doc in documents]

    # TF-IDF Vectorisation
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(corpus)
    query_tfidf = tfidf.transform([query])

    # Similarité TF-IDF (cosine)
    cosine_scores = (query_tfidf @ tfidf_matrix.T).toarray()[0]
    top_idx = np.argmax(cosine_scores)
    top_doc = documents[top_idx]

    # Vectorisation dense pour CosineSim@1
    query_vec = model.encode(query)
    top_vec = model.encode(top_doc["text"])
    cosine_sim_1 = 1 - cosine(query_vec, top_vec)

    # LangMatch
    query_lang = detect(query)
    lang_match = 1 if query_lang == top_doc["lang"] else 0

    # TF-IDF Score
    tfidf_score = cosine_scores[top_idx]

    # Temps de traitement
    duration = time.time() - start_time

    # Ragas-like Score (pondération arbitraire)
    ragas_like = 0.4 * cosine_sim_1 + 0.3 * tfidf_score + 0.2 * lang_match + 0.1 * (1 - duration)

    result = {
        "top_document": top_doc["text"],
        "top_document_lang": top_doc["lang"],
        "CosineSim@1": round(cosine_sim_1, 4),
        "TF-IDF score": round(tfidf_score, 4),
        "LangMatch": lang_match,
        "Query Time (s)": round(duration, 4),
        "Ragas-like Score": round(ragas_like, 4),
    }

    return result

In [52]:
def get_all_documents(client, collection_name, output_fields, batch_size=1000):
    all_results = []
    offset = 0
    max_limit = 16384  # Limite imposée par Milvus

    while True:
        # Ajuster le batch_size pour ne pas dépasser la limite
        current_limit = min(batch_size, max_limit - offset)
        if current_limit <= 0:
            break

        results = client.query(
            collection_name=collection_name,
            offset=offset,
            limit=current_limit,
            output_fields=output_fields
        )
        if not results:
            break

        all_results.extend(results)
        offset += current_limit
        print(f"Récupérés : {offset} documents")

    return pd.DataFrame(all_results)

In [ ]:
output_fields = ["id", "text", "lang", "vector"]
documents = get_all_documents(client, collection, output_fields)

Récupérés : 1000 documents


In [ ]:
query = "Qui est Carlos Sandov"

In [ ]:
res = exact_match_retrieval_tfidf(query, documents, model)
res = pd.DataFrame(res)

res.to_csv("exact_match_retrieval_tfidf.csv", index=False)